In [1]:
from wav_to_embedding import * 
import tqdm

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/torch/_utils.py:776: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


In [2]:
from wav_to_embedding import wav_to_embedding, get_wav_files
import tqdm
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

data_directory = "data"

wav_files = get_wav_files(data_directory)

hubert_embeddings = []
for file_path, label in tqdm.tqdm(wav_files, desc="Processing WAV files"):
    # Obtain the embedding for the current WAV file
    embedding_tensors = wav_to_embedding(file_path)
    
    # Flatten each layer's embedding and store them separately
    for layer_num, embedding_tensor in enumerate(embedding_tensors):
        embedding = embedding_tensor.detach().numpy().flatten()
        # Append the file name, label, and flattened embedding to the data list
        hubert_embeddings.append((file_path, label, layer_num, embedding))

# Convert the data list to a DataFrame
hubert_embeddings_df = pd.DataFrame(hubert_embeddings, columns=['Soundtrack', 'Label', 'Layer', 'Embedding'])

# Create a DataFrame to store results
results_df = pd.DataFrame({'Actual_Label': [], 'Predicted_Label': [], 'Layer': []})

# Iterate through each layer
for layer_num in tqdm.tqdm(hubert_embeddings_df['Layer'].unique(), desc="Layer-wise classification"):
    # Clone the original DataFrame to preserve the original embeddings
    phonation_mode_df = hubert_embeddings_df[hubert_embeddings_df['Layer'] == layer_num].copy(deep=True)
    # Extract embeddings for the current layer
    max_length = max(len(embedding) for embedding in phonation_mode_df['Embedding'])
    X = np.array([np.pad(embedding, (0, max_length - len(embedding))) for embedding in phonation_mode_df['Embedding']])
    y = phonation_mode_df['Label']

    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Train SVM classifier
    svm_classifier = SVC(kernel='linear')  # Linear kernel works well for high-dimensional data
    svm_classifier.fit(X_train, y_train)

    # Evaluate classifier
    y_pred = svm_classifier.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy for Layer {layer_num}: {accuracy}")

    # Create a DataFrame to store results for the current layer
    layer_results_df = pd.DataFrame({
        'Actual_Label': y_test,
        'Predicted_Label': y_pred,
        'Layer': layer_num,
        'Layer_Accuracy': accuracy
    })
    
    # Concatenate the results with the overall results_df
    results_df = pd.concat([results_df, layer_results_df], ignore_index=True)
    
# Merge results with hubert_embeddings_df and fill NA for Predicted_Label
merged_df = pd.merge(hubert_embeddings_df, results_df, left_on=['Layer', 'Label'], right_on=['Layer', 'Actual_Label'], how='left')
merged_df['Predicted_Label'] = merged_df['Predicted_Label'].fillna(np.nan)

# Save merged DataFrame to CSV
merged_df.to_csv('merged_results.csv', index=False)
merged_df

Layer-wise classification:   4%|▋               | 1/25 [00:17<06:52, 17.19s/it]

Accuracy for Layer 0: 0.9150326797385621


Layer-wise classification:   8%|█▎              | 2/25 [00:35<06:44, 17.59s/it]

Accuracy for Layer 1: 0.9150326797385621


Layer-wise classification:  12%|█▉              | 3/25 [00:54<06:46, 18.46s/it]

...

Layer-wise classification:  96%|██████████████▍| 24/25 [09:08<00:23, 23.83s/it]

Accuracy for Layer 23: 0.8300653594771242


Layer-wise classification: 100%|███████████████| 25/25 [09:31<00:00, 22.85s/it]

Accuracy for Layer 24: 0.8169934640522876


,Soundtrack,Label,Layer,Embedding,Actual_Label,Predicted_Label,Layer_Accuracy
0,data/Pressed/a4_I_pressed_norm.wav,Pressed,0,"[3.798892, 34.334892, -0.89043355, 8.472467, 2...",Pressed,Pressed,0.915033
1,data/Pressed/a4_I_pressed_norm.wav,Pressed,0,"[3.798892, 34.334892, -0.89043355, 8.472467, 2...",Pressed,Pressed,0.915033
2,data/Pressed/a4_I_pressed_norm.wav,Pressed,0,"[3.798892, 34.334892, -0.89043355, 8.472467, 2...",Pressed,Pressed,0.915033
3,data/Pressed/a4_I_pressed_norm.wav,Pressed,0,"[3.798892, 34.334892, -0.89043355, 8.472467, 2...",Pressed,Pressed,0.915033
4,data/Pressed/a4_I_pressed_norm.wav,Pressed,0,"[3.798892, 34.334892, -0.89043355, 8.472467, 2...",Pressed,Pressed,0.915033
...,...,...,...,...,...,...,...
784320,data/Flow/F4#_E_flow_norm.wav,Flow,24,"[0.2015838, 0.66090417, 0.1571862, 0.17119673,...",Flow,Flow,0.816993
784321,data/Flow/F4#_E_flow_norm.wav,Flow,24,"[0.2015838, 0.66090417, 0.1571862, 0.17119673,...",Flow,Flow,0.816993
784322,data/Flow/F4#_E_flow_norm.wav,Flow,24,"[0.2015838, 0.66090417, 0.1571862, 0.17119673,...",Flow,Flow,0.816993
784323,data/Flow/F4#_E_flow_norm.wav,Flow,24,"[0.2015838, 0.66090417, 0.1571862, 0.17119673,...",Flow,Flow,0.816993


## Trimming instead of padding

In [5]:
from wav_to_embedding import wav_to_embedding, get_wav_files
import tqdm
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

data_directory = "data"

wav_files = get_wav_files(data_directory)

hubert_embeddings = []
for file_path, label in tqdm.tqdm(wav_files, desc="Processing WAV files"):
    # Obtain the embedding for the current WAV file
    embedding_tensors = wav_to_embedding(file_path)
    
    # Flatten each layer's embedding and store them separately
    for layer_num, embedding_tensor in enumerate(embedding_tensors):
        embedding = embedding_tensor.detach().numpy().flatten()
        
        if len(embedding) < 80000:
            embedding = np.pad(embedding, (0, 80000 - len(embedding)))
        else:
            embedding = embedding[:80000]
            
        # Append the file name, label, and flattened embedding to the data list
        hubert_embeddings.append((file_path, label, layer_num, embedding))

# Convert the data list to a DataFrame
hubert_embeddings_df = pd.DataFrame(hubert_embeddings, columns=['Soundtrack', 'Label', 'Layer', 'Embedding'])

# Create a DataFrame to store results
results_df = pd.DataFrame({'Actual_Label': [], 'Predicted_Label': [], 'Layer': []})

# Iterate through each layer
for layer_num in tqdm.tqdm(hubert_embeddings_df['Layer'].unique(), desc="Layer-wise classification"):
    # Clone the original DataFrame to preserve the original embeddings
    phonation_mode_df = hubert_embeddings_df[hubert_embeddings_df['Layer'] == layer_num].copy(deep=True)
    # Extract embeddings for the current layer
    max_length = max(len(embedding) for embedding in phonation_mode_df['Embedding'])
    X = np.array(list(phonation_mode_df['Embedding']))
    y = phonation_mode_df['Label']

    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Train SVM classifier
    svm_classifier = SVC(kernel='linear')  # Linear kernel works well for high-dimensional data
    svm_classifier.fit(X_train, y_train)

    # Evaluate classifier
    y_pred = svm_classifier.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"Accuracy for Layer {layer_num}: {accuracy}")

    # Create a DataFrame to store results for the current layer
    layer_results_df = pd.DataFrame({
        'Actual_Label': y_test,
        'Predicted_Label': y_pred,
        'Layer': layer_num,
        'Layer_Accuracy': accuracy
    })
    
    # Concatenate the results with the overall results_df
    results_df = pd.concat([results_df, layer_results_df], ignore_index=True)
    
# Merge results with hubert_embeddings_df and fill NA for Predicted_Label
merged_df = pd.merge(hubert_embeddings_df, results_df, left_on=['Layer', 'Label'], right_on=['Layer', 'Actual_Label'], how='left')
merged_df['Predicted_Label'] = merged_df['Predicted_Label'].fillna(np.nan)

# Save merged DataFrame to CSV
merged_df.to_csv('merged_results_em_trim.csv', index=False)
merged_df

Layer-wise classification:   4%|▋               | 1/25 [00:13<05:30, 13.75s/it]

Accuracy for Layer 0: 0.9150326797385621


Layer-wise classification:   8%|█▎              | 2/25 [00:27<05:22, 14.01s/it]

Accuracy for Layer 1: 0.9215686274509803


Layer-wise classification:  12%|█▉              | 3/25 [00:43<05:22, 14.68s/it]

...

Layer-wise classification:  96%|██████████████▍| 24/25 [08:07<00:18, 18.86s/it]

Accuracy for Layer 23: 0.8169934640522876


Layer-wise classification: 100%|███████████████| 25/25 [08:25<00:00, 20.22s/it]

Accuracy for Layer 24: 0.8169934640522876


,Soundtrack,Label,Layer,Embedding,Actual_Label,Predicted_Label,Layer_Accuracy
0,data/Pressed/a4_I_pressed_norm.wav,Pressed,0,"[3.798892, 34.334892, -0.89043355, 8.472467, 2...",Pressed,Pressed,0.915033
1,data/Pressed/a4_I_pressed_norm.wav,Pressed,0,"[3.798892, 34.334892, -0.89043355, 8.472467, 2...",Pressed,Pressed,0.915033
2,data/Pressed/a4_I_pressed_norm.wav,Pressed,0,"[3.798892, 34.334892, -0.89043355, 8.472467, 2...",Pressed,Pressed,0.915033
3,data/Pressed/a4_I_pressed_norm.wav,Pressed,0,"[3.798892, 34.334892, -0.89043355, 8.472467, 2...",Pressed,Pressed,0.915033
4,data/Pressed/a4_I_pressed_norm.wav,Pressed,0,"[3.798892, 34.334892, -0.89043355, 8.472467, 2...",Pressed,Pressed,0.915033
...,...,...,...,...,...,...,...
784320,data/Flow/F4#_E_flow_norm.wav,Flow,24,"[0.2015838, 0.66090417, 0.1571862, 0.17119673,...",Flow,Flow,0.816993
784321,data/Flow/F4#_E_flow_norm.wav,Flow,24,"[0.2015838, 0.66090417, 0.1571862, 0.17119673,...",Flow,Flow,0.816993
784322,data/Flow/F4#_E_flow_norm.wav,Flow,24,"[0.2015838, 0.66090417, 0.1571862, 0.17119673,...",Flow,Flow,0.816993
784323,data/Flow/F4#_E_flow_norm.wav,Flow,24,"[0.2015838, 0.66090417, 0.1571862, 0.17119673,...",Flow,Flow,0.816993


## 5-fold cross validation

In [6]:
from wav_to_embedding import wav_to_embedding, get_wav_files
import tqdm
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

data_directory = "data"

wav_files = get_wav_files(data_directory)

hubert_embeddings = []
for file_path, label in tqdm.tqdm(wav_files, desc="Processing WAV files"):
    # Obtain the embedding for the current WAV file
    embedding_tensors = wav_to_embedding(file_path)
    
    # Flatten each layer's embedding and store them separately
    for layer_num, embedding_tensor in enumerate(embedding_tensors):
        embedding = embedding_tensor.detach().numpy().flatten()
        # Append the file name, label, and flattened embedding to the data list
        hubert_embeddings.append((file_path, label, layer_num, embedding))

# Convert the data list to a DataFrame
hubert_embeddings_df = pd.DataFrame(hubert_embeddings, columns=['Soundtrack', 'Label', 'Layer', 'Embedding'])

# Create a DataFrame to store results
results_df = pd.DataFrame({'Actual_Label': [], 'Predicted_Label': [], 'Layer': [], 'Layer_Accuracy': []})

# Iterate through each layer
for layer_num in tqdm.tqdm(hubert_embeddings_df['Layer'].unique(), desc="Layer-wise classification"):
    # Clone the original DataFrame to preserve the original embeddings
    phonation_mode_df = hubert_embeddings_df[hubert_embeddings_df['Layer'] == layer_num].copy(deep=True)
    # Extract embeddings for the current layer
    max_length = max(len(embedding) for embedding in phonation_mode_df['Embedding'])
    X = np.array([np.pad(embedding, (0, max_length - len(embedding))) for embedding in phonation_mode_df['Embedding']])
    y = phonation_mode_df['Label']

    # Train SVM classifier with 5-fold cross-validation
    svm_classifier = SVC(kernel='linear')  # Linear kernel works well for high-dimensional data
    cv_scores = cross_val_score(svm_classifier, X, y, cv=5)

    # Print average accuracy across folds
    print(f"Average accuracy for Layer {layer_num}: {np.mean(cv_scores)}")

    # Store cross-validation results
    for fold_num, accuracy in enumerate(cv_scores):
        fold_results_df = pd.DataFrame({
            'Actual_Label': y,
            'Predicted_Label': None,  # No predictions in cross-validation
            'Layer': layer_num,
            'Fold': fold_num,
            'Fold_Accuracy': accuracy
        })
        results_df = pd.concat([results_df, fold_results_df], ignore_index=True)

# Save results to CSV
results_df.to_csv('cross_validation_results.csv', index=False)
results_df

Layer-wise classification:   4%|▋               | 1/25 [01:22<32:59, 82.48s/it]

Average accuracy for Layer 0: 0.9251891984864121


Layer-wise classification:   8%|█▎              | 2/25 [02:53<33:26, 87.26s/it]

Average accuracy for Layer 1: 0.930452356381149


Layer-wise classification:  12%|█▉              | 3/25 [04:30<33:41, 91.89s/it]

...

Layer-wise classification:  96%|███████████▌| 24/25 [1:57:46<02:28, 148.49s/it]

Average accuracy for Layer 23: 0.7926126590987272


Layer-wise classification: 100%|████████████| 25/25 [1:59:38<00:00, 287.15s/it]

Average accuracy for Layer 24: 0.7965514275885793


,Actual_Label,Predicted_Label,Layer,Layer_Accuracy,Fold,Fold_Accuracy
0,Pressed,None,0.0,NaN,0.0,0.947712
1,Pressed,None,0.0,NaN,0.0,0.947712
2,Pressed,None,0.0,NaN,0.0,0.947712
3,Pressed,None,0.0,NaN,0.0,0.947712
4,Pressed,None,0.0,NaN,0.0,0.947712
...,...,...,...,...,...,...
95245,Flow,None,24.0,NaN,4.0,0.815789
95246,Flow,None,24.0,NaN,4.0,0.815789
95247,Flow,None,24.0,NaN,4.0,0.815789
95248,Flow,None,24.0,NaN,4.0,0.815789


## Dev archive

Experi
- different classifier
- wav2vec-base
- wav2vec-large

In [ ]:
experi setup
experiments
results

ETA
- histogram of the length of the wav files by phonation mode
- consider trimmings instead of padding

- histo on wave freq and amplitude - and norm

HuBERT
- https://www.sciencedirect.com/science/article/pii/S0885230823000694
- investigate earlier hidden_states

Wav2Vec
- Try as replacement for HuBERT - wav2vec large and base
- investigate earlier hidden_states

In [11]:
data_directory = "unit_test_data"

wav_files = get_wav_files(data_directory)

hubert_embeddings = []
for file_path, label in tqdm.tqdm(wav_files, desc="Processing WAV files"):
    # Obtain the embedding for the current WAV file
    embedding_tensor = wav_to_embedding(file_path)
    
    # Flatten the embedding tensor
    embedding = embedding_tensor.detach().numpy().flatten()  # Convert tensor to numpy array
    
    # Append the file name, label, and flattened embedding to the data list
    hubert_embeddings.append((file_path, label, embedding))

# Convert the data list to a DataFrame
hubert_embeddings_df = pd.DataFrame(hubert_embeddings, columns=['Soundtrack', 'Label', 'Embedding'])

# Save DataFrame to CSV
hubert_embeddings_df.to_csv('hubert_embeddings.csv', index=False)

Processing WAV files: 100%|██████████████████████| 4/4 [00:00<00:00,  5.90it/s]


In [12]:
hubert_embeddings_df

,Soundtrack,Label,Embedding
0,unit_test_data/Pressed/A3_A_pressedta_norm.wav,Pressed,"[0.19391328, 0.5167813, 0.13981955, 0.17416999..."
1,unit_test_data/Breathy/A3_A_breathy_2_norm.wav,Breathy,"[0.085249886, 0.4596952, 0.3463359, 0.1114618,..."
2,unit_test_data/Normal/A3_A_neutral_norm.wav,Normal,"[0.1342236, 0.61969876, 0.24281624, 0.11822165..."
3,unit_test_data/Flow/A3_A_flow_norm.wav,Flow,"[0.13961063, 0.6294041, 0.21230346, 0.1959945,..."


In [13]:
data_directory = "data"

wav_files = get_wav_files(data_directory)

hubert_embeddings = []
for file_path, label in tqdm.tqdm(wav_files, desc="Processing WAV files"):
    # Obtain the embedding for the current WAV file
    embedding_tensor = wav_to_embedding(file_path)
    
    # Flatten the embedding tensor
    embedding = embedding_tensor.detach().numpy().flatten()  # Convert tensor to numpy array
    # do i need to normalize the tensor embeddings?
    
    # Append the file name, label, and flattened embedding to the data list
    hubert_embeddings.append((file_path, label, embedding))

# Convert the data list to a DataFrame
hubert_embeddings_df = pd.DataFrame(hubert_embeddings, columns=['Soundtrack', 'Label', 'Embedding'])

Processing WAV files: 100%|██████████████████| 762/762 [01:58<00:00,  6.42it/s]


In [16]:
phonation_mode_df = hubert_embeddings_df.copy(deep=True)

In [18]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Step 1: Extract features and labels

# X = np.stack(phonation_mode_df['Embedding'])  # Convert string representation of array to numpy array
# y = phonation_mode_df['Label']

# Pad or truncate embeddings to a fixed length
max_length = max(len(embedding) for embedding in phonation_mode_df['Embedding'])
X = np.array([np.pad(embedding, (0, max_length - len(embedding))) for embedding in phonation_mode_df['Embedding']])

# Extract labels
y = phonation_mode_df['Label']

- do 5-fold CV - find correct CV algo from sklearn

In [19]:
# Step 2: Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# normalize the test/train data - ETA to check for variance

# Step 3: Train SVM classifier
svm_classifier = SVC(kernel='linear')  # Linear kernel works well for high-dimensional data
svm_classifier.fit(X_train, y_train)

SVC(kernel='linear')

In [20]:
# Step 4: Evaluate classifier
y_pred = svm_classifier.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

Accuracy: 0.8169934640522876


In [21]:
# Create a DataFrame to store results
results_df = pd.DataFrame({
    'Actual_Label': y_test,
    'Predicted_Label': y_pred,
    'Features': X_test.tolist()
})

In [31]:
results_df

,Actual_Label,Predicted_Label,Features
196,Pressed,Pressed,"[0.21948839724063873, 0.7289494872093201, 0.16..."
260,Pressed,Pressed,"[0.20392000675201416, 0.5029839277267456, 0.06..."
39,Pressed,Pressed,"[0.1743122786283493, 0.5102345943450928, 0.068..."
449,Breathy,Breathy,"[0.2533986270427704, 0.49114271998405457, 0.15..."
595,Normal,Normal,"[0.20222653448581696, 0.5663732290267944, 0.13..."
...,...,...,...
550,Normal,Normal,"[0.20668986439704895, 0.5305796265602112, 0.16..."
382,Breathy,Breathy,"[0.259615033864975, 0.49446117877960205, 0.204..."
412,Breathy,Breathy,"[0.3085710406303406, 0.6310077905654907, 0.132..."
181,Pressed,Pressed,"[0.16483867168426514, 0.5255987048149109, 0.02..."


- Grid search for hyper param tuning

In [30]:
results_df[results_df['Actual_Label']!=results_df['Predicted_Label']]

,Actual_Label,Predicted_Label,Features
479,Normal,Breathy,"[0.2036820650100708, 0.5917807817459106, 0.171..."
446,Breathy,Pressed,"[0.1821623593568802, 0.6138148903846741, 0.112..."
576,Normal,Breathy,"[0.25455325841903687, 0.572446346282959, 0.151..."
342,Breathy,Pressed,"[0.22416679561138153, 0.504161536693573, 0.127..."
231,Pressed,Breathy,"[0.2095051407814026, 0.6910457611083984, 0.165..."
756,Flow,Pressed,"[0.25982820987701416, 0.5880413055419922, 0.07..."
275,Pressed,Breathy,"[0.29546836018562317, 0.5168625712394714, 0.28..."
617,Normal,Flow,"[0.2186421900987625, 0.5753768086433411, 0.120..."
718,Flow,Pressed,"[0.16297554969787598, 0.6142287254333496, 0.13..."
728,Flow,Normal,"[0.16142725944519043, 0.7253645658493042, 0.16..."


In [ ]:
from transformers import AutoProcessor, HubertModel
import soundfile as sf
import numpy as np
import pandas as pd
import librosa
import os

processor = AutoProcessor.from_pretrained("facebook/hubert-large-ls960-ft")
model = HubertModel.from_pretrained("facebook/hubert-large-ls960-ft")

def get_wav_files(directory, debug=False):
    wav_files = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith(".wav"):
                # Extract the label (subfolder name)
                label = os.path.basename(root)
                # Construct the full path to the WAV file
                file_path = os.path.join(root, file)
                # Append tuple containing file path and label to list
                wav_files.append((file_path, label))
    if debug:
        # Print file paths and corresponding labels
        for file_path, label in wav_files:
            print(f"File Path: {file_path}, Label: {label}")
    return wav_files

def resample_audio(audio, original_sr, target_sr):
    return librosa.resample(audio, orig_sr=original_sr, target_sr=target_sr)
    # librosa - normalizing = audio/max_absolute (make every signal -1 to 1 range norm)

def map_to_array(file_path):
    track, sample_rate = sf.read(file_path)
    if sample_rate != 16000:  # If the sample rate is not 16 kHz, resample it
        track = resample_audio(track, sample_rate, 16000)
        sample_rate = 16000
    return track, sample_rate

def wav_to_embedding(file_path):
    # Load each WAV file, map it to an array and its sample rate
    track, sample_rate = map_to_array(file_path)

    # Preprocess each array and convert it into input values
    input_value = processor(track, sampling_rate=sample_rate, return_tensors="pt").input_values

    # Pass the input values through the model to get the hidden states
    hidden_state = model(input_value).last_hidden_state

    return hidden_state